# 23 — LightGBM (L1, L2 y promedio de 3 modelos)

**En el material:** LightGBM solo aparece **nombrado** en `material/Arboles/Árboles.pdf` (slide 65, el logo), entre XGBoost (64), Gradient Boosting (66) y Histogram Gradient Boosting (67). No hay notebook de clase que lo use, así que acá se explica desde cero.

**Qué es.** Un Gradient Boosting (slide 66: árboles chicos que se suman, cada uno corrige los errores de los anteriores) que, como el HistGB (slide 67), agrupa cada variable en a lo sumo 255 cajas (histogramas) antes de buscar cortes. El `HistGradientBoostingClassifier` de sklearn está inspirado en LightGBM.

**Diferencias que importan acá:**
1. **Categóricas nativas con muchos valores.** LightGBM acepta `seller_id` (~28.000 valores) y `category_id` como categóricas: en cada corte ordena las categorías por cuánto empujan hacia `new` y busca el mejor punto de corte sobre ese orden. El HistGB de sklearn solo acepta categóricas de hasta 255 valores. Para no sobreajustar en categorías chicas: `min_data_per_group` (mínimo de filas por categoría para tenerla en cuenta; 100 por defecto) y `cat_smooth` (acerca las categorías chicas al promedio; 10 por defecto).
2. **Crecimiento del árbol:** abre primero la hoja que más mejora el error, limitado por `num_leaves` = 31. El HistGB hace lo mismo con `max_leaf_nodes` = 31, así que acá no hay diferencia.
3. **Sin *early stopping* por defecto:** 100 árboles fijos. El HistGB usa *early stopping* automático con más de 10.000 filas (frena en ~110-120 árboles; `12_tuning`).

**Pruebas:**
- **L1:** el pipeline del HistGB P7a con duración (`armar_pipeline_histgb_combos()`), cambiando **solo** el modelo por `LGBMClassifier` por defecto. Comparación pareada contra P7a con duración (0,9412).
- **L2:** las mismas entradas que el CatBoost P6b con duración (numéricas + duración, TF-IDF 500, `prob_texto_new`, 6 categóricas crudas), con las 6 categóricas como **categóricas nativas**, sin target encoding. Tres configuraciones: por defecto, `min_data_per_group`/`cat_smooth` = 200/50 y 500/100.
- **Promedio de 3 modelos** (CatBoost + HistGB + LightGBM) contra el modelo final (0,9420): pesos iguales, y pesos elegidos con 4 folds y evaluados en el quinto.

**No es leakage:** todo lo que aprende de los datos va dentro del `Pipeline` y se ajusta solo con el train de cada fold. En L2, el `OrdinalEncoder` (que convierte cada categoría en un número entero) se aprende en el fold; las categorías que no vio en ese fold quedan como "faltante". Las categóricas nativas sí miran el target al ordenar categorías, pero solo con las filas de entrenamiento del árbol, igual que cualquier corte de un árbol.

Las probabilidades *out-of-fold* (OOF) de cada prueba se guardan, así el promedio con CatBoost y HistGB se arma sin reentrenar (las de los modelos finales están en `.cache_laboratorio/oof_p7c_duracion.csv`, semilla 42).

In [1]:
import sys
sys.path.append("..")
import os, time, warnings
import numpy as np
import pandas as pd
from joblib import Memory
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OrdinalEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_predict

from features import (aplanar_anidadas, agregar_features_contenido, agregar_combos, agregar_logs,
                      armar_pipeline_histgb_combos, armar_modelo_texto, PrediccionOOF, CapeoPorMoneda,
                      COLUMNAS_NUMERICAS, COLUMNAS_CATEGORICAS)

SEED = 42
crudo = pd.read_json("../data/train_data.jsonlines", lines=True)
y = (crudo["condition"] == "new").astype(int)
train = agregar_combos(agregar_features_contenido(aplanar_anidadas(crudo)))

carpeta_lab = ".cache_laboratorio"
memoria_texto = Memory(location=carpeta_lab, verbose=0)

# Referencias de la 18 (semilla 42): OOF de CatBoost y HistGB P7a, los dos con duración
oof_ref = pd.read_csv(os.path.join(carpeta_lab, "oof_p7c_duracion.csv"))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = np.empty(len(train), dtype=int)
for k, (_, val) in enumerate(cv.split(train, y)):
    folds[val] = k + 1
assert (folds == oof_ref["fold"].to_numpy()).all() and (y.to_numpy() == oof_ref["y"].to_numpy()).all()


def acc_por_fold(prob, folds_=folds):
    return pd.Series((prob >= 0.5).astype(int) == y.to_numpy()).groupby(folds_).mean().to_numpy()


def comparar(acc, ref):
    diferencias = np.round(acc - ref, 10)
    gana = int((diferencias > 0).sum())
    return {"media": acc.mean(), "dif_media": diferencias.mean(), "dif_desvio": diferencias.std(), "folds_gana": gana,
            "pasa": gana == 5 or (gana == 4 and diferencias.mean() > diferencias.std())}


acc_histgb = acc_por_fold(oof_ref["histgb"].to_numpy())
acc_final = acc_por_fold(0.5 * oof_ref["catboost"].to_numpy() + 0.5 * oof_ref["histgb"].to_numpy())
print("HistGB P7a + duración:", acc_histgb.round(4), acc_histgb.mean().round(4))
print("Modelo final (50/50): ", acc_final.round(4), acc_final.mean().round(4))

HistGB P7a + duración: [0.9428 0.9393 0.9426 0.9394 0.9422] 0.9412
Modelo final (50/50):  [0.9431 0.9389 0.9424 0.942  0.9436] 0.942


In [2]:
def con_texto_cacheado(pipeline):
    # Reemplaza el modelo de texto por la versión cacheada del laboratorio (mismo resultado, más rápido)
    transformers = pipeline.named_steps["preprocesador"].transformers
    i = [n for n, _, _ in transformers].index("texto")
    transformers[i] = ("texto", Pipeline([("oof", PrediccionOOF(armar_modelo_texto(), random_state=SEED)),
                                          ("identidad", "passthrough")], memory=memoria_texto), ["title", "warranty_texto"])
    return pipeline


def lgbm(**parametros):
    # n_jobs=2: los 5 folds corren en paralelo (5 x 2 = 10 de los 12 núcleos)
    return LGBMClassifier(random_state=SEED, verbose=-1, n_jobs=2, **parametros)


def oof(pipeline, params_fit=None):
    inicio = time.time()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        prob = cross_val_predict(pipeline, train, y, cv=cv, method="predict_proba", n_jobs=5, params=params_fit or {})[:, 1]
    return prob, (time.time() - inicio) / 60


resultados, probs = {}, {}

## L1 — mismo pipeline que el HistGB P7a, con LightGBM como modelo

Lo único que cambia es el último paso del `Pipeline` (`set_params(modelo=...)`): mismas features (one-hot de 4 categóricas, numéricas, TF-IDF, `prob_texto_new`, target encoding de vendedor, categoría y 19 combinaciones, conteo y precio del vendedor, duración). Si da casi lo mismo que el HistGB, confirma que son el mismo algoritmo con otra implementación.

In [3]:
pipeline_l1 = con_texto_cacheado(armar_pipeline_histgb_combos()).set_params(modelo=lgbm())
probs["L1"], minutos = oof(pipeline_l1)
resultados["L1 - LightGBM con features de P7a (vs HistGB P7a)"] = {**comparar(acc_por_fold(probs["L1"]), acc_histgb), "minutos": minutos}
pd.DataFrame(resultados).T.round(4)

,media,dif_media,dif_desvio,folds_gana,pasa,minutos
L1 - LightGBM con features de P7a (vs HistGB P7a),0.940857,-0.000386,0.000724,1,False,0.379131


## L2 — categóricas nativas (las entradas del CatBoost P6b)

`OrdinalEncoder` convierte cada categoría en un entero (0, 1, 2, …), aprendido en el train del fold; las no vistas quedan en −1, que LightGBM trata como faltante. Las 6 categóricas van primero en el `ColumnTransformer` (índices 0-5), y se le indican a LightGBM con `categorical_feature` en el `fit`.

Se prueban configuraciones de `min_data_per_group` / `cat_smooth`: por defecto, 200/50 y 500/100 (plan original: suavizar más), y después de ver que suavizar **empeora**, 20/1 y 5/1 (con `max_cat_threshold` = 256: cuántas categorías puede mandar a un lado en un corte; 32 por defecto). **Sesgo de selección:** elegir la mejor de 3 sobre los mismos folds la favorece un poco; por eso lo que pase se confirma con la semilla 7.

In [4]:
def armar_l2(**parametros):
    preprocesador = ColumnTransformer([
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1,
                               encoded_missing_value=-1), COLUMNAS_CATEGORICAS),
        ("num", "passthrough", COLUMNAS_NUMERICAS + ["duracion_dias"]),
        ("tfidf", TfidfVectorizer(max_features=500), "title"),
        ("texto", PrediccionOOF(armar_modelo_texto(), random_state=SEED), ["title", "warranty_texto"]),
    ], sparse_threshold=0)
    pipeline = Pipeline([
        ("capeo", CapeoPorMoneda(columnas=["price", "base_price"], columna_grupo="currency_id")),
        ("logs", FunctionTransformer(agregar_logs)),
        ("preprocesador", preprocesador),
        ("modelo", lgbm(**parametros)),
    ])
    return con_texto_cacheado(pipeline)


INDICES_CAT_L2 = list(range(len(COLUMNAS_CATEGORICAS)))
configuraciones = {"L2a (por defecto: 100 / 10)": {},
                   "L2b (200 / 50)": {"min_data_per_group": 200, "cat_smooth": 50},
                   "L2c (500 / 100)": {"min_data_per_group": 500, "cat_smooth": 100},
                   # Agregadas después de ver que suavizar más empeora: el sentido contrario
                   "L2d (20 / 1)": {"min_data_per_group": 20, "cat_smooth": 1},
                   "L2e (5 / 1, max_cat_threshold 256)": {"min_data_per_group": 5, "cat_smooth": 1, "max_cat_threshold": 256}}
for nombre, parametros in configuraciones.items():
    probs[nombre], minutos = oof(armar_l2(**parametros), {"modelo__categorical_feature": INDICES_CAT_L2})
    resultados[f"{nombre} - categóricas nativas (vs HistGB P7a)"] = {**comparar(acc_por_fold(probs[nombre]), acc_histgb), "minutos": minutos}
    print(nombre, acc_por_fold(probs[nombre]).mean().round(4), f"{minutos:.1f} min")
pd.DataFrame(resultados).T.round(4)

L2a (por defecto: 100 / 10) 0.9324 1.3 min


L2b (200 / 50) 0.9259 0.4 min


L2c (500 / 100) 0.9229 0.3 min


L2d (20 / 1) 0.932 0.3 min


L2e (5 / 1, max_cat_threshold 256) 0.9263 0.3 min


,media,dif_media,dif_desvio,folds_gana,pasa,minutos
L1 - LightGBM con features de P7a (vs HistGB P7a),0.940857,-0.000386,0.000724,1,False,0.379131
L2a (por defecto: 100 / 10) - categóricas nativas (vs HistGB P7a),0.932386,-0.008857,0.002316,0,False,1.288075
L2b (200 / 50) - categóricas nativas (vs HistGB P7a),0.925943,-0.0153,0.00252,0,False,0.388082
L2c (500 / 100) - categóricas nativas (vs HistGB P7a),0.922929,-0.018314,0.00247,0,False,0.322994
L2d (20 / 1) - categóricas nativas (vs HistGB P7a),0.931971,-0.009271,0.002682,0,False,0.263695
"L2e (5 / 1, max_cat_threshold 256) - categóricas nativas (vs HistGB P7a)",0.926329,-0.014914,0.001601,0,False,0.261335


## Promedio de 3 modelos contra el modelo final (0,9420)

Con las OOF ya calculadas: CatBoost + HistGB (de la 18) + el mejor LightGBM.
- **Pesos iguales** (1/3 cada uno).
- **Pesos elegidos con 4 folds y evaluados en el quinto**, rotando: grilla de pesos de a 0,1 que suman 1; se elige la de mejor accuracy en los otros 4 folds (igual que en `16_ensambles`).

In [5]:
mejor_l = max(probs, key=lambda k: acc_por_fold(probs[k]).mean())
p_cat, p_hgb, p_lgb = oof_ref["catboost"].to_numpy(), oof_ref["histgb"].to_numpy(), probs[mejor_l]
print("LightGBM elegido:", mejor_l, "| correlación con CatBoost / HistGB:",
      np.corrcoef(p_lgb, p_cat)[0, 1].round(4), np.corrcoef(p_lgb, p_hgb)[0, 1].round(4))

acc_iguales = acc_por_fold((p_cat + p_hgb + p_lgb) / 3)
resultados["3 modelos, pesos iguales (vs modelo final)"] = comparar(acc_iguales, acc_final)

grilla = [(a / 10, b / 10, (10 - a - b) / 10) for a in range(11) for b in range(11 - a)]
prob_pond, pesos_elegidos = np.empty(len(train)), {}
for k in range(1, 6):
    entrena, evalua = folds != k, folds == k
    mejor = max(grilla, key=lambda w: (((w[0] * p_cat + w[1] * p_hgb + w[2] * p_lgb)[entrena] >= 0.5) == y.to_numpy()[entrena]).mean())
    pesos_elegidos[k] = mejor
    prob_pond[evalua] = (mejor[0] * p_cat + mejor[1] * p_hgb + mejor[2] * p_lgb)[evalua]
print("Pesos (CatBoost, HistGB, LightGBM) elegidos por fold:", pesos_elegidos)
resultados["3 modelos, pesos elegidos en 4 folds (vs modelo final)"] = comparar(acc_por_fold(prob_pond), acc_final)

pd.DataFrame({"fold": folds, "y": y, **{k: v for k, v in probs.items()}}).to_csv(
    os.path.join(carpeta_lab, "oof_lightgbm.csv"), index=False)
pd.DataFrame(resultados).T.round(4)

LightGBM elegido: L1 | correlación con CatBoost / HistGB: 0.9932 0.9978
Pesos (CatBoost, HistGB, LightGBM) elegidos por fold: {1: (0.6, 0.4, 0.0), 2: (0.6, 0.4, 0.0), 3: (0.3, 0.5, 0.2), 4: (0.3, 0.7, 0.0), 5: (0.6, 0.4, 0.0)}


,media,dif_media,dif_desvio,folds_gana,pasa,minutos
L1 - LightGBM con features de P7a (vs HistGB P7a),0.940857,-0.000386,0.000724,1,False,0.379131
L2a (por defecto: 100 / 10) - categóricas nativas (vs HistGB P7a),0.932386,-0.008857,0.002316,0,False,1.288075
L2b (200 / 50) - categóricas nativas (vs HistGB P7a),0.925943,-0.0153,0.00252,0,False,0.388082
L2c (500 / 100) - categóricas nativas (vs HistGB P7a),0.922929,-0.018314,0.00247,0,False,0.322994
L2d (20 / 1) - categóricas nativas (vs HistGB P7a),0.931971,-0.009271,0.002682,0,False,0.263695
"L2e (5 / 1, max_cat_threshold 256) - categóricas nativas (vs HistGB P7a)",0.926329,-0.014914,0.001601,0,False,0.261335
"3 modelos, pesos iguales (vs modelo final)",0.941886,-0.000114,0.000308,1,False,NaN
"3 modelos, pesos elegidos en 4 folds (vs modelo final)",0.9417,-0.0003,0.00051,2,False,NaN


## Conclusión

| Prueba | Accuracy | Contra | Diferencia (folds) |
|---|---|---|---|
| L1: LightGBM con las features de P7a | 0,9409 | HistGB P7a + duración (0,9412) | −0,0004 (1/5) |
| L2a: categóricas nativas, por defecto (100/10) | 0,9324 | HistGB P7a + duración | −0,0089 (0/5) |
| L2b / L2c: más suavizado (200/50, 500/100) | 0,9259 / 0,9229 | ídem | −0,015 / −0,018 (0/5) |
| L2d / L2e: menos suavizado (20/1, 5/1) | 0,9320 / 0,9263 | ídem | −0,009 / −0,015 (0/5) |
| CatBoost + HistGB + L1, pesos iguales | 0,9419 | modelo final (0,9420) | −0,0001 (1/5) |
| CatBoost + HistGB + L1, pesos elegidos en 4 folds | 0,9417 | modelo final | −0,0003 (2/5); L1 recibe 0 en 4 de 5 folds |

- **L1 ≈ HistGB**, como se esperaba (los dos son Gradient Boosting con histogramas, slides 66-67). Correlación de probabilidades con el HistGB: 0,998. No aporta diversidad al promedio.
- **Las categóricas nativas de LightGBM rinden mucho peor** (0,923-0,932) que el target encoding con *cross-fitting* del HistGB (0,9412) y que las *ordered target statistics* de CatBoost (0,9414 con las mismas entradas). Suavizar más o menos no lo arregla.
- **Interpretación (no verificada):** con ~28.000 vendedores, LightGBM ordena las categorías en cada corte usando el target de las **mismas** filas con las que entrena el árbol, sin *cross-fitting* ni orden aleatorio. Los vendedores chicos quedan "memorizados". El `TargetEncoder` y CatBoost evitan justamente eso: cada fila se codifica sin usar su propia etiqueta. Es un buen argumento para la presentación: **cómo se codifica `seller_id` importa más que qué boosting se use.**
- **Decisión:** no se cambia nada. Modelo final = P7c + duración (0,9420). OOF guardadas en `.cache_laboratorio/oof_lightgbm.csv`.